# Spreadsheets end to end

Loading an `.xlsx` with `read_excel`, working on it as a DataFrame, then
handing the numeric columns to the natural-language layer.

## Before running

From the repo root:

```sh
pip install -e ".[all,dev]"
pip install python-dotenv
```

`[all]` brings the provider SDKs. Installing only `[excel,dev]` leaves the
model cells failing with *"Please install google-genai"*, because the
provider package ships separately from `pydantic-ai` itself.

Then put a key in `examples/.env`, matching whichever model you use:

```
GEMINI_API_KEY=...
```

Cells up to *Asking questions* need no key and no provider SDK.

In [1]:
from dotenv import load_dotenv

load_dotenv(".env")

import numpy as np
import numpyai_dashboard as npi

## Loading

`read_excel` returns a `pandas.DataFrame`. Every column is kept, with its type
inferred by the Rust reader.

In [2]:
df = npi.read_excel("sample_sales.xlsx")
df.head()

,region,rep,product,order_date,units,unit_price,discount,closed,notes
0,EMEA,R. Ahmed,Basic,2023-01-01,21.0,19.99,NaN,False,renewal
1,APAC,R. Brown,Pro,2023-01-03,26.0,49.50,0.20,True,NaN
2,AMER,R. Chen,Enterprise,2023-01-05,5.0,199.00,0.25,True,NaN
3,LATAM,R. Duarte,Basic,2023-01-07,7.0,19.99,0.11,False,NaN
4,EMEA,R. Eriksen,Pro,2023-01-09,4.0,49.50,0.27,True,renewal


In [3]:
df.dtypes

region                   str
rep                      str
product                  str
order_date    datetime64[ms]
units                float64
unit_price           float64
discount             float64
closed                  bool
notes                    str
dtype: object

Text stays text, dates become `datetime64`, `TRUE`/`FALSE` become `bool`, and
blank cells become the null of whichever type the column is. Nothing is dropped.

In [4]:
df[["discount", "notes"]].isna().sum()

discount     14
notes       112
dtype: int64

### Options

Pick a sheet by name or index, skip the header row, or read only the first
`n_rows` when you just want to look at the shape of a large file.

In [5]:
npi.read_excel("sample_sales.xlsx", sheet="Sales", n_rows=5)

,region,rep,product,order_date,units,unit_price,discount,closed,notes
0,EMEA,R. Ahmed,Basic,2023-01-01,21.0,19.99,NaN,False,renewal
1,APAC,R. Brown,Pro,2023-01-03,26.0,49.50,0.20,True,NaN
2,AMER,R. Chen,Enterprise,2023-01-05,5.0,199.00,0.25,True,NaN
3,LATAM,R. Duarte,Basic,2023-01-07,7.0,19.99,0.11,False,NaN
4,EMEA,R. Eriksen,Pro,2023-01-09,4.0,49.50,0.27,True,renewal


With `header=False` the first row is data and columns are positional.

In [6]:
npi.read_excel("sample_sales.xlsx", header=False).head(3)

,col0,col1,col2,col3,col4,col5,col6,col7,col8
0,region,rep,product,order_date,units,unit_price,discount,closed,notes
1,EMEA,R. Ahmed,Basic,2023-01-01 00:00:00,21,19.99,NaN,false,renewal
2,APAC,R. Brown,Pro,2023-01-03 00:00:00,26,49.5,0.2,true,NaN


Sheets can be chosen by index too, and unsupported formats fail loudly
rather than half-working.

In [7]:
from numpyai_dashboard._exceptions import NumpyAIError

print(npi.read_excel("sample_sales.xlsx", sheet=0).shape)

try:
    npi.read_excel("notes.csv")
except NumpyAIError as exc:
    print("error:", exc)

(150, 9)
error: read_excel cannot read .csv files. Use read_csv for delimited text.


## Working in pandas

Ordinary DataFrame work. Note `discount` has blanks, so fill them before
arithmetic.

In [8]:
df["revenue"] = df["units"] * df["unit_price"] * (1 - df["discount"].fillna(0))

df.groupby("region")["revenue"].sum().sort_values(ascending=False).round(2)

region
APAC     92209.21
EMEA     84916.77
LATAM    81800.69
AMER     75051.67
Name: revenue, dtype: float64

`order_date` is a real `datetime64`, so date filtering works without parsing.

In [9]:
q1 = df[df["order_date"] < "2023-04-01"]
print(f"{len(q1)} orders in Q1")
q1.groupby("product")["revenue"].sum().round(2)

45 orders in Q1


product
Basic          6284.66
Enterprise    77178.17
Pro           18788.72
Name: revenue, dtype: float64

## Asking questions

Every column is in scope, so questions can name any of them.

## Asking questions

Everything below needs a provider key. Generated code is syntax-checked and
independently judged before a result comes back.

In [ ]:
df.chat("Total revenue by region since March.")

`chat` returns a `ChatResult`: the answer in `.value`, plus the code
that produced it and the judge's verdict.

In [ ]:
result = df.chat("Which product line has the highest average discount?")

print("value      :", result.value)
print("attempts   :", result.attempts)
print("ok         :", result.ok)
print("description:", result.description)
print("code:")
print(result.code)

In [ ]:
df.chat("Correlation between units and revenue.")

Missing values are visible to the model, so it can be asked about them directly.

In [ ]:
df.chat("How many rows have a missing discount?")

Questions can group and filter on text and date columns, not just numbers.

In [ ]:
df.chat("Mean units per rep, for closed orders only.")

A query can return an array rather than a scalar.

In [ ]:
imputed = df.chat("Replace missing discounts with the column mean.")
imputed.value

## Several inputs at once

`NumpyAISession` takes arrays and DataFrames, exposed as `arr1`, `df2`, ...

In [ ]:
emea = df.data[df["region"] == "EMEA"]
apac = df.data[df["region"] == "APAC"]

sess = npi.NumpyAISession([emea, apac])
sess.chat("Compare the mean revenue of the two tables.")

## Diagnosis

Suggests analysis steps for the data rather than computing an answer.

In [20]:
diag = npi.Diagnosis(sess)
diag.steps(task="Give me 5 steps to analyse this sales data.")

────────────────────────────────────────────────── LLM Response ───────────────────────────────────────────────────

╭────────────────────────────────────────────── Data Analysis Steps ──────────────────────────────────────────────╮
│ [ "1. Initial Data Cleaning and Missing Value Imputation: Begin by inspecting both 'arr1' and 'arr2' for        │
│ missing values (NaNs), specifically in the discount column (column index 2) as indicated by the metadata. Use a │
│ NumPy function to count NaNs per column to understand the extent of missing data. For diagnosis, if the         │
│ percentage of missing values in a column is low (e.g., <5-10%), consider imputing them using the column's       │
│ median or mean (NumPy's nanmean or nanmedian) to preserve data distribution and avoid introducing bias from     │
│ extreme values. If missing values are extensive, investigate the possibility of dropping rows with NaNs or more │
│ sophisticated imputation methods. The output of this step will be two clean arrays, 'arr1_clean' and            │
│ 'arr2_clean', suitable for numerical operations.", "2. Descriptive Statistics and Outlier Identification:       │
│ Calculate key descriptive statistics for each column of the cleaned arrays ('arr1_clean' and 'arr2_clean'),     │
│ such as mean, median, standard deviation, minimum, and maximum using relevant NumPy functions. Pay close        │
│ attention to the total sales column (column index 3) and the quantity/price columns (indices 0 and 1). For      │
│ diagnosis, compare these statistics across both arrays to identify any significant differences or anomalies.    │
│ Additionally, identify potential outliers, especially in the total sales column, by using methods like the      │
│ Interquartile Range (IQR) rule (e.g., values outside 1.5 * IQR from the quartiles) or Z-scores, which can be    │
│ computed using NumPy's percentile and std functions. The output will be a statistical summary for each feature  │
│ and a list of identified outliers.", "3. Feature Relationship Analysis and Engineering: Investigate the         │
│ relationships between the features within each cleaned array. For instance, calculate the correlation matrix    │
│ using a NumPy function to understand how quantity, price, and discount relate to total sales. For diagnosis,    │
│ strong correlations can indicate important drivers of sales. Additionally, consider simple feature engineering, │
│ such as verifying if total sales is directly a product of quantity and price adjusted by discount (if not,      │
│ create such a feature), or calculating price per unit. This step helps in understanding underlying patterns and │
│ preparing potentially more informative features for subsequent analysis. The output will be correlation         │
│ matrices and any newly engineered features appended to the arrays.", "4. Product Segment Analysis: Observe that │
│ the second column (index 1) has distinct price categories (e.g., 19.99, 49.50, 199.00). Segment the data based  │
│ on these price categories for each array. For each segment, calculate aggregate statistics like the average     │
│ total sales, average discount applied, and average quantity sold using NumPy's boolean indexing and aggregation │
│ functions (mean, sum). For diagnosis, this allows for comparison of sales performance and discount              │
│ effectiveness across different product tiers or types. For example, does a higher price category correlate with │
│ different discount usage or sales volume? The output will be a summarized view of sales performance per price   │
│ segment.", "5. Comparative Analysis and Simple Predictive Modeling: Compare the overall performance and         │
│ distributions of features between 'arr1' and 'arr2' based on the insights from previous steps. Identify which   │
│ array represents higher average sales, more frequent discounts, or different product mixes. For diagnosis, this │
│ step helps in understanding if the datasets represent different market segments or time periods. Further, as an │
│ initial predictive step, using the quantity, price, an

["1. **Initial Data Cleaning and Missing Value Imputation**: Begin by inspecting both 'arr1' and 'arr2' for missing values (NaNs), specifically in the discount column (column index 2) as indicated by the metadata. Use a NumPy function to count NaNs per column to understand the extent of missing data. For diagnosis, if the percentage of missing values in a column is low (e.g., <5-10%), consider imputing them using the column's median or mean (NumPy's `nanmean` or `nanmedian`) to preserve data distribution and avoid introducing bias from extreme values. If missing values are extensive, investigate the possibility of dropping rows with NaNs or more sophisticated imputation methods. The output of this step will be two clean arrays, 'arr1_clean' and 'arr2_clean', suitable for numerical operations.",
 "2. **Descriptive Statistics and Outlier Identification**: Calculate key descriptive statistics for each column of the cleaned arrays ('arr1_clean' and 'arr2_clean'), such as mean, median, stan

## Verbose mode

`verbose=True` prints every intermediate step: the generated code, the
judgement, and any retries.

In [ ]:
loud = npi.read_excel("sample_sales.xlsx", verbose=True)
loud.chat("How many orders are closed?")

## Choosing a model

Any Pydantic AI model spec works. The default is `google:gemini-2.5-flash`.

In [ ]:
# needs the matching extra installed, e.g. numpyai-dashboard[anthropic]
# alt = npi.read_excel("sample_sales.xlsx", model="anthropic:claude-sonnet-4-5")
# alt.chat("Median units.")

## The judge rejects non-answers

Ask for something the data cannot support and the model will often hand back a
polite explanation rather than a computation. The judge is what stops that from
counting as an answer: it compares the generated code against the query and
rejects prose standing in for a result.

With `max_tries=1` there is no retry, so `chat` prints the failures, warns, and
returns `None`. It does not raise, so one bad question will not end a session.
Raise `max_tries` to watch it retry with the rejection fed back in.

In [ ]:
stubborn = npi.read_excel("sample_sales.xlsx", max_tries=1)
stubborn.chat("What is the average customer age?")